# How Coding Agents Work: Claude Code and Cursor, from Their Documentation

The runnable code of [How Coding Agents Work: Claude Code and Cursor, from Their Documentation](https://residual-stream.lvtan71.workers.dev/posts/how-coding-agents-work/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/how-coding-agents-work.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 1 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "how-coding-agents-work"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in []:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## Subagents and hooks

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/how-coding-agents-work/#subagents-and-hooks)

In [ ]:
%%writefile guard.py
#!/usr/bin/env python3
# A PreToolUse hook for Claude Code: it receives the pending tool call as
# JSON on stdin; exit code 2 blocks the call, and what it prints to stderr
# is shown to the model as the reason. Exit 0 lets the call proceed.
# A check like this shows the mechanism; it is not a security boundary (a
# command can always be written another way).
import json
import re
import shlex
import sys

def dangerous(command):
    try:
        words = shlex.split(command)
    except ValueError:
        return False
    for i, w in enumerate(words):
        args = words[i + 1:]
        flags = "".join(a.lstrip("-") for a in args if a.startswith("-") and not a.startswith("--"))
        long = {a for a in args if a.startswith("--")}
        if w == "rm" and ("r" in flags.lower() or "--recursive" in long) and ("f" in flags or "--force" in long):
            return True                                          # rm -rf, -fr, -r -f, -Rf, --recursive --force
        if w == "push" and i > 0 and words[i - 1] == "git":
            if "f" in flags or "--force" in long or any(re.match(r"\+", a) for a in args):
                return True                                      # force push (--force-with-lease is allowed)
    return False

event = json.load(sys.stdin)
if event.get("tool_name") == "Bash" and dangerous(event.get("tool_input", {}).get("command", "")):
    print("Blocked by project policy: recursive force delete or force push. Ask the user first.", file=sys.stderr)
    sys.exit(2)
sys.exit(0)

In [ ]:
import json
import subprocess
import sys

# Feed the hook pending tool calls, as Claude Code would.
for command in ("npm test", "rm -rf build/", "rm -fr build/", "rm -r -f build/", "rm build/old.log",
                "git push --force-with-lease", "git push -f origin main", "python -c 'import shutil; shutil.rmtree(\"build\")'"):
    event = {"hook_event_name": "PreToolUse", "tool_name": "Bash", "tool_input": {"command": command}}
    r = subprocess.run([sys.executable, "guard.py"], input=json.dumps(event), capture_output=True, text=True)
    print(f"{command!r:58s} -> {'allowed' if r.returncode == 0 else 'blocked (exit 2)'}")

The post printed:

```text
'npm test'                                                 -> allowed
'rm -rf build/'                                            -> blocked (exit 2)
'rm -fr build/'                                            -> blocked (exit 2)
'rm -r -f build/'                                          -> blocked (exit 2)
'rm build/old.log'                                         -> allowed
'git push --force-with-lease'                              -> allowed
'git push -f origin main'                                  -> blocked (exit 2)
'python -c \'import shutil; shutil.rmtree("build")\''      -> allowed
```